# ICT305 Assignment 2

**Topic:** Factors that shape Singapore's happiness (we argue *against* the CNA 3rd-happiest-city claim).

The most of the happiness datasets are annual (income, hours worked, deaths). So the master is **yearly**, with monthly/quarterly data either aggregated up to a yearly mean OR kept as standalone side-tables for charts that need higher resolution.

Output files (saved to `data/processed/`):
1. `master_yearly.csv` one row per year, columns from every member
2. `cpi_monthly.csv` full CPI monthly series (for M2's inflation chart)
3. `hdb_resale_monthly.csv` full HDB transactions Jan 2017 onwards (for M3's price story)
4. `hdb_price_index_quarterly.csv` HDB resale price index quarterly (for M3's trend chart)
5. `cause_of_death_2024_long.csv` long-format death data ready for M5's headline chart
6. `mccy_youth_happiness.csv` happiness Likert scores for under-35 vs 35+ groups

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
print('ROOT      :', ROOT)
print('RAW       :', RAW)
print('PROCESSED :', PROCESSED)

ROOT      : C:\Users\MYP\Desktop\Parthiban\Uni\CS_AI\2026_Term_5\ICT305\Assignment\ICT3025 - Group Assignment 2
RAW       : C:\Users\MYP\Desktop\Parthiban\Uni\CS_AI\2026_Term_5\ICT305\Assignment\ICT3025 - Group Assignment 2\data\raw
PROCESSED : C:\Users\MYP\Desktop\Parthiban\Uni\CS_AI\2026_Term_5\ICT305\Assignment\ICT3025 - Group Assignment 2\data\processed


## Section 1： Inventory check

In [2]:
for member in ['member_1', 'member_2', 'member_3', 'member_4', 'member_5']:
    folder = RAW / member
    files = sorted(folder.glob('*.csv')) if folder.exists() else []
    print(f'{member:<10} : {len(files)} files')
    for f in files:
        print(f'   {f.name:<60} {f.stat().st_size/1024:>9,.1f} KB')

member_1   : 5 files
   MOM_Usual_Hours_Worked_Annual.csv                                  0.3 KB
   SingStat_Labour_Force_Participation_Rate.csv                      61.0 KB
   SingStat_LongTerm_Unemployment_Rate_Annual.csv                    17.4 KB
   SingStat_Resident_Unemployment_Rate_Annual.csv                    29.9 KB
   SingStat_Workplace_Injury_Rate_Annual.csv                          1.5 KB
member_2   : 7 files
   CPI_General_and_Healthcare.csv                                     0.7 KB
   SingStat_CPI_Monthly_2024Base.csv                                203.6 KB
   SingStat_CPI_YoY_Annual.csv                                      118.6 KB
   SingStat_GDP_Per_Capita_Annual.csv                                15.4 KB
   SingStat_Gini_Coefficient_2000_present.csv                         6.6 KB
   SingStat_Key_Indicators_Household_Employment_Income_Annual.csv      76.8 KB
   SingStat_Key_Indicators_Household_Market_Income_Annual.csv        51.2 KB
member_3   : 4 files
   HDB_Medi

## Section 2： M1 Mobility / Work-Life Balance (annual)

MOM file is wide (year columns 2010-2025). Melt into long, then keep two yearly columns: mean weekly hours and percent of workers doing 40+ hours.

In [3]:
mom = pd.read_csv(RAW / 'member_1' / 'MOM_Usual_Hours_Worked_Annual.csv')
mom_long = mom.melt(id_vars='DataSeries', var_name='year', value_name='value')
mom_long['year']  = pd.to_numeric(mom_long['year'], errors='coerce').astype('Int64')
mom_long['value'] = pd.to_numeric(mom_long['value'], errors='coerce')

# Pivot the two series into their own columns
m1_yearly = (
    mom_long.pivot_table(index='year', columns='DataSeries', values='value', aggfunc='mean')
    .rename(columns={
        'Mean Usual Hours Worked Per Week': 'm1_mean_weekly_hours',
        'Percentage Of Workers Who Usually Worked At Least 40 Hours Per Week': 'm1_pct_40plus_hours',
    })
    .reset_index()
)
print('M1 yearly:', m1_yearly.shape)
m1_yearly.head()

M1 yearly: (16, 3)


DataSeries,year,m1_mean_weekly_hours,m1_pct_40plus_hours
0,2010,46.6,88.3
1,2011,45.9,87.3
2,2012,45.6,87.4
3,2013,45.3,87.1
4,2014,44.3,85.6


## Section 3： M2 Cost of Living (annual aggregates)

We aggregate monthly CPI to annual averages here, but also save the raw monthly file as a side-table for the dedicated inflation chart that M2 will make.

In [4]:
# CPI Monthly (full series, kept as side-table)
cpi_m = pd.read_csv(RAW / 'member_2' / 'SingStat_CPI_Monthly_2024Base.csv')
cpi_m['value'] = pd.to_numeric(cpi_m['value'], errors='coerce')
cpi_m['month'] = pd.to_datetime(cpi_m['period'], format='%Y %b', errors='coerce')
cpi_m = cpi_m.dropna(subset=['month'])
cpi_m.to_csv(PROCESSED / 'cpi_monthly.csv', index=False)
print('CPI monthly side-table:', cpi_m.shape)

# Aggregate to annual mean for the master
cpi_m['year'] = cpi_m['month'].dt.year
FOCUS_ITEMS = ['All Items', 'Food', 'Housing & Utilities', 'Transport']
cpi_y = (
    cpi_m[cpi_m['series'].isin(FOCUS_ITEMS)]
    .groupby(['year', 'series'])['value'].mean()
    .unstack('series')
    .add_prefix('m2_cpi_')
    .reset_index()
)
cpi_y.columns = [c.replace(' & ', '_').replace(' ', '_').lower() for c in cpi_y.columns]
print('CPI yearly aggregate:', cpi_y.shape)
cpi_y.head()

CPI monthly side-table: (5000, 5)
CPI yearly aggregate: (66, 3)


,year,m2_cpi_all_items,m2_cpi_food
0,1961,20.953333,18.454417
1,1962,21.045667,18.579083
2,1963,21.494000,19.263000
3,1964,21.848333,19.745583
4,1965,21.901500,19.648417


In [5]:
# Household income annual (median income series)
inc = pd.read_csv(RAW / 'member_2' / 'SingStat_Key_Indicators_Household_Employment_Income_Annual.csv')
inc['value']  = pd.to_numeric(inc['value'], errors='coerce')
inc['year']   = pd.to_numeric(inc['period'], errors='coerce').astype('Int64')

TARGET_INCOME_SERIES = [
    'Median Monthly Household Employment Income Including Employer CPF Contributions',
    'Real Change In Median Monthly Household Employment Income Including Employer CPF Contributions',
]
income_y = (
    inc[inc['series'].isin(TARGET_INCOME_SERIES)]
    .pivot_table(index='year', columns='series', values='value', aggfunc='mean')
    .rename(columns={
        TARGET_INCOME_SERIES[0]: 'm2_median_household_income_sgd',
        TARGET_INCOME_SERIES[1]: 'm2_real_income_change_pct',
    })
    .reset_index()
)
print('Income yearly:', income_y.shape)
income_y.head()

Income yearly: (26, 3)


series,year,m2_median_household_income_sgd,m2_real_income_change_pct
0,2000,4400.0,NaN
1,2001,4719.0,6.8
2,2002,4595.0,-2.4
3,2003,4617.0,-0.1
4,2004,4556.0,-2.8


In [6]:
# Gini coefficient (annual)
gini = pd.read_csv(RAW / 'member_2' / 'SingStat_Gini_Coefficient_2000_present.csv')
gini['year']  = pd.to_numeric(gini['period'], errors='coerce').astype('Int64')
gini['value'] = pd.to_numeric(gini['value'], errors='coerce')

GINI_INCLUDE = gini[gini['series'].str.contains('Including', case=False, na=False) &
                    gini['series'].str.contains('Member', case=False, na=False)]
gini_y = (
    GINI_INCLUDE.groupby('year')['value'].mean()
    .rename('m2_gini_coefficient').reset_index()
)
print('Gini yearly:', gini_y.shape)
gini_y.head()

Gini yearly: (26, 2)


,year,m2_gini_coefficient
0,2000,0.4280
1,2001,0.4365
2,2002,0.4340
3,2003,0.4395
4,2004,0.4395


In [7]:
# GDP per capita (annual, SGD)， the "headline prosperity" number
# Used to show the GDP-vs-actual-income gap (top earners pull the average way up).
gdp = pd.read_csv(RAW / 'member_2' / 'SingStat_GDP_Per_Capita_Annual.csv')
gdp['value'] = pd.to_numeric(gdp['value'], errors='coerce')
gdp['year']  = pd.to_numeric(gdp['period'], errors='coerce').astype('Int64')

gdp_sgd_y = (
    gdp[gdp['series'] == 'Per Capita Gross Domestic Product']
    .groupby('year')['value']
    .mean()
    .rename('m2_gdp_per_capita_sgd')
    .reset_index()
)

# Compute the income-per-person gap
# Singapore average resident household size = 3.11 (SingStat).
HH_SIZE = 3.11
gap = income_y.merge(gdp_sgd_y, on='year', how='outer')
gap['m2_income_per_person_sgd'] = gap['m2_median_household_income_sgd'] / HH_SIZE
gap['m2_gdp_vs_income_gap_x']   = (gap['m2_gdp_per_capita_sgd'] / 12) / gap['m2_income_per_person_sgd']

gdp_y = gap[['year', 'm2_gdp_per_capita_sgd', 'm2_income_per_person_sgd', 'm2_gdp_vs_income_gap_x']]

print('GDP per capita yearly:', gdp_y.shape)
gdp_y.tail()

GDP per capita yearly: (66, 4)


,year,m2_gdp_per_capita_sgd,m2_income_per_person_sgd,m2_gdp_vs_income_gap_x
61,2021,108667.0,3068.810289,2.950845
62,2022,125773.0,3254.019293,3.220965
63,2023,115993.0,3499.035370,2.762499
64,2024,126803.0,3637.942122,2.904641
65,2025,129194.0,3867.202572,2.783968


## Section 4： M3 Housing (quarterly to yearly aggregates)

HDB Resale Price Index is quarterly. We aggregate to yearly mean for the master, and keep the full quarterly series as a side-table for M3's trend chart. The huge resale-flat-prices file is too large for the master, so we just save it as a side-table.

In [8]:
# HDB Resale Price Index (quarterly to yearly)
rpi = pd.read_csv(RAW / 'member_3' / 'HDB_Resale_Price_Index_Quarterly.csv')
rpi['year'] = rpi['quarter'].str.slice(0, 4).astype(int)
rpi.to_csv(PROCESSED / 'hdb_price_index_quarterly.csv', index=False)

rpi_y = (
    rpi.groupby('year')['index'].mean()
    .rename('m3_hdb_resale_price_index').reset_index()
)
print('HDB price index yearly:', rpi_y.shape)
rpi_y.tail()

HDB price index yearly: (37, 2)


,year,m3_hdb_resale_price_index
32,2022,165.850
33,2023,177.175
34,2024,190.600
35,2025,202.800
36,2026,203.400


In [9]:
# HDB Resale Flat Prices (transactions, monthly)， save as side-table only
# This is 22 MB, too granular for the master
hdb_t = pd.read_csv(RAW / 'member_3' / 'HDB_Resale_Flat_Prices_Jan2017_onwards.csv')
print('HDB transactions:', hdb_t.shape)

# Save unmodified as side-table
hdb_t.to_csv(PROCESSED / 'hdb_resale_monthly.csv', index=False)

# Derive yearly aggregates (median price + transaction count) for the master
hdb_t['resale_price'] = pd.to_numeric(hdb_t['resale_price'], errors='coerce')
hdb_t['year'] = pd.to_datetime(hdb_t['month']).dt.year

hdb_y = (
    hdb_t.groupby('year')['resale_price']
    .agg(m3_hdb_median_resale_price='median',
         m3_hdb_resale_transactions='count')
    .reset_index()
)
print('HDB transactions yearly:', hdb_y.shape)
hdb_y

HDB transactions: (233479, 11)
HDB transactions yearly: (10, 3)


,year,m3_hdb_median_resale_price,m3_hdb_resale_transactions
0,2017,410000.0,20509
1,2018,408000.0,21561
2,2019,400000.0,22186
3,2020,425000.0,23333
4,2021,483000.0,29087
5,2022,525000.0,26720
6,2023,550000.0,25754
7,2024,590000.0,27832
8,2025,628000.0,25085
9,2026,629000.0,11412


## Section 5： M5 Healthcare / Youth (annual, derived)

We compute the headline metric (% of deaths by suicide for each age band, 2024) and save it as a side-table that M5 will plot directly. We also keep a long-format death file for M5's other charts.

In [10]:
# Headline: suicide share of all deaths by age, 2024
deaths = pd.read_csv(RAW / 'member_5' / 'Cause_of_death_by_age_2024.csv')
deaths['DEATH_COUNT'] = pd.to_numeric(deaths['DEATH_COUNT'], errors='coerce')

# Long-format save for M5's other charts
deaths.to_csv(PROCESSED / 'cause_of_death_2024_long.csv', index=False)

# Compute the killer metric: suicide share by age band
suicide_mask = deaths['ICD_DETAILED_CATEGORY'].str.contains('Intentional self-harm', case=False, na=False)

by_age_total   = deaths.groupby('DEATH_AGE')['DEATH_COUNT'].sum().rename('total_deaths')
by_age_suicide = deaths[suicide_mask].groupby('DEATH_AGE')['DEATH_COUNT'].sum().rename('suicide_deaths')

youth_table = pd.concat([by_age_total, by_age_suicide], axis=1).fillna(0)
youth_table['suicide_share_pct'] = 100 * youth_table['suicide_deaths'] / youth_table['total_deaths']
youth_table = youth_table.sort_values('suicide_share_pct', ascending=False).reset_index()
youth_table.to_csv(PROCESSED / 'suicide_share_by_age_2024.csv', index=False)
print('Suicide share by age 2024 (killer metric for M5):')
print(youth_table.to_string(index=False))

Suicide share by age 2024 (killer metric for M5):
      DEATH_AGE  total_deaths  suicide_deaths  suicide_share_pct
          10-19            87            22.0          25.287356
        20 - 29           223            47.0          21.076233
        30 - 39           436            75.0          17.201835
        40 - 49           837            42.0           5.017921
        50 - 59          1818            37.0           2.035204
        60 - 64          1751            22.0           1.256425
        65 - 69          2621            23.0           0.877528
        70 - 74          3187            15.0           0.470662
        75 - 79          3491            15.0           0.429676
        80 - 84          3880             8.0           0.206186
        85 - 89          3867             4.0           0.103439
    90 and Over          4130             4.0           0.096852
              2             2             0.0           0.000000
              1             9           

In [11]:
# Principal Causes of Death (2006-2022 annual) — pivot to wide, keep only top causes
pcod = pd.read_csv(RAW / 'member_5' / 'Principal_Causes_of_Death_2006_2022.csv')
pcod['percentage_deaths'] = pd.to_numeric(pcod['percentage_deaths'], errors='coerce')

# Normalise condition names (case differences across years)
pcod['disease_condition'] = pcod['disease_condition'].str.title()

TOP_CAUSES = ['Cancer', 'Ischaemic Heart Disease', 'Pneumonia',
              'Cerebrovascular Disease (Including Stroke)', 'External Causes Of Morbidity And Mortality']
pcod_y = (
    pcod[pcod['disease_condition'].isin(TOP_CAUSES)]
    .pivot_table(index='year', columns='disease_condition', values='percentage_deaths', aggfunc='mean')
    .add_prefix('m5_pct_deaths_')
    .reset_index()
)
pcod_y.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').lower() for c in pcod_y.columns]
print('Principal Causes yearly:', pcod_y.shape)
pcod_y.head()

Principal Causes yearly: (17, 6)


,year,m5_pct_deaths_cancer,m5_pct_deaths_cerebrovascular_disease_including_stroke,m5_pct_deaths_external_causes_of_morbidity_and_mortality,m5_pct_deaths_ischaemic_heart_disease,m5_pct_deaths_pneumonia
0,2006,28.5,8.9,NaN,18.5,13.7
1,2007,27.7,8.7,NaN,19.8,13.9
2,2008,29.3,8.3,NaN,20.1,13.9
3,2009,29.3,8.0,NaN,19.2,15.3
4,2010,28.5,8.4,NaN,18.7,15.7


In [12]:
# MCCY happiness Likert scores by age， youth-vs-older comparison
mccy = pd.read_csv(RAW / 'member_5' / 'MCCY_Social_Values_Survey_2018_2019.csv', low_memory=False)

# `outcome_connection` and `outcome_future` are 0-10 Likert; force to numeric
for c in ['outcome_connection', 'outcome_future', 'support_immedfam', 'support_friends']:
    if c in mccy.columns:
        mccy[c + '_num'] = pd.to_numeric(
            mccy[c].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
        )

happiness_by_age = (
    mccy.groupby('age_2')[[c + '_num' for c in
                            ['outcome_connection', 'outcome_future', 'support_immedfam', 'support_friends']
                            if c + '_num' in mccy.columns]]
    .mean()
    .reset_index()
)
happiness_by_age.to_csv(PROCESSED / 'mccy_youth_happiness.csv', index=False)
print('MCCY happiness Likert (mean by age band):')
print(happiness_by_age.to_string(index=False))

MCCY happiness Likert (mean by age band):
          age_2  outcome_connection_num  outcome_future_num  support_immedfam_num  support_friends_num
16-19 years old                7.611940            7.511194              8.395522             8.294776
20-24 years old                7.454976            7.170616              8.255924             7.630332
25-34 years old                7.474609            7.296875              8.419922             7.632812
35-44 years old                7.613806            7.347015              8.242537             7.171642
45-54 years old                7.680734            7.313761              8.264220             6.908257
55-64 years old                7.708487            6.937269              7.977860             6.378229
65-75 years old                8.064935            6.987013              7.887446             6.324675


## Section 5b： M4 Sleep / Stress / World Happiness

Three M4 files: the WHR panel (16 years of Singapore happiness scores), the WHR 2024 ranking snapshot (143 countries), and a global sleep/stress dataset (no Singapore data, kept for the occupation-level comparison only).

We pull two yearly columns into the master from the WHR panel: Singapore's Life Ladder score and Freedom To Make Life Choices. Both directly contradict the "happy city" claim once you compare them with peer countries.

In [13]:
# M4: pull Singapore happiness panel (yearly) from the WHR updated 2024 file
whr = pd.read_csv(RAW / 'member_4' / 'World-happiness-report-updated_2024.csv', encoding='latin-1')

# Save the full panel as a side-table so M4 can compare Singapore vs other countries
whr.to_csv(PROCESSED / 'whr_panel_full.csv', index=False)
print('WHR panel side-table:', whr.shape)

# Extract Singapore-only series for the master yearly merge
sg = whr[whr['Country name'] == 'Singapore'].copy()
sg = sg.rename(columns={
    'year': 'year',
    'Life Ladder': 'm4_life_ladder',
    'Freedom to make life choices': 'm4_freedom_score',
    'Social support': 'm4_social_support',
    'Positive affect': 'm4_positive_affect',
    'Negative affect': 'm4_negative_affect',
})
m4_yearly = sg[['year', 'm4_life_ladder', 'm4_freedom_score',
                'm4_social_support', 'm4_positive_affect', 'm4_negative_affect']]
m4_yearly['year'] = pd.to_numeric(m4_yearly['year'], errors='coerce').astype('Int64')
m4_yearly = m4_yearly.dropna(subset=['year']).reset_index(drop=True)

print('M4 Singapore yearly:', m4_yearly.shape, '  span', m4_yearly['year'].min(), 'to', m4_yearly['year'].max())
m4_yearly.head()

WHR panel side-table: (2363, 11)
M4 Singapore yearly: (16, 6)   span 2006 to 2023


,year,m4_life_ladder,m4_freedom_score,m4_social_support,m4_positive_affect,m4_negative_affect
0,2006,6.463,0.757,0.904,0.689,0.267
1,2007,6.834,0.867,0.921,0.588,0.114
2,2008,6.642,0.661,0.845,0.627,0.256
3,2009,6.145,0.776,0.866,0.450,0.208
4,2010,6.531,0.846,0.864,0.527,0.131


In [14]:
# M4 side-table: WHR 2024 snapshot ranking (143 countries) for the "Singapore vs Copenhagen/Zurich" chart
whr_snap = pd.read_csv(RAW / 'member_4' / 'World-happiness-report-2024.csv')
whr_snap.to_csv(PROCESSED / 'whr_2024_ranking.csv', index=False)
print('WHR 2024 snapshot:', whr_snap.shape)

# Peer-city quick check for the report
HAPPY_CITIES_PEERS = ['Singapore','Denmark','Switzerland','Finland','South Korea','Sweden','Norway']
peers = whr_snap[whr_snap['Country name'].isin(HAPPY_CITIES_PEERS)].copy()
peers = peers.sort_values('Ladder score', ascending=False)
print('\nWHR 2024 score for peers (Copenhagen=Denmark, Zurich=Switzerland, Seoul=South Korea):')
print(peers[['Country name','Ladder score','Social support','Freedom to make life choices','Generosity']].to_string(index=False))

# M4 side-table: Sleep dataset (no Singapore but useful for global occupation stress comparison)
sleep = pd.read_csv(RAW / 'member_4' / 'Sleep_health_and_lifestyle_dataset.csv')
sleep.to_csv(PROCESSED / 'sleep_health_global.csv', index=False)
print('\nSleep dataset side-table:', sleep.shape, '   (note: no Singapore rows)')

WHR 2024 snapshot: (143, 12)

WHR 2024 score for peers (Copenhagen=Denmark, Zurich=Switzerland, Seoul=South Korea):
Country name  Ladder score  Social support  Freedom to make life choices  Generosity
     Finland         7.741           1.572                         0.859       0.142
     Denmark         7.583           1.520                         0.823       0.204
      Sweden         7.344           1.501                         0.838       0.221
      Norway         7.302           1.517                         0.835       0.224
 Switzerland         7.060           1.425                         0.759       0.173
   Singapore         6.523           1.361                         0.743       0.168
 South Korea         6.058           1.178                         0.555       0.126

Sleep dataset side-table: (374, 13)    (note: no Singapore rows)


## Section 6： Master yearly merge

Outer-merge every yearly slice on `year`, then trim to a sensible window (2010 onwards is what makes sense for the happiness story most series start around then).

In [15]:
yearly_frames = {
    'M1 mobility'        : m1_yearly,
    'M2 cpi'             : cpi_y,
    'M2 income'          : income_y,
    'M2 gini'            : gini_y,
    'M2 gdp gap'         : gdp_y,
    'M3 hdb price index' : rpi_y,
    'M3 hdb transactions': hdb_y,
    'M4 WHR Singapore'   : m4_yearly,
    'M5 principal causes': pcod_y,
}

master = None
for label, df in yearly_frames.items():
    df = df.copy()
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['year'])
    print(f'{label:<22} : {df.shape}')
    master = df if master is None else master.merge(df, on='year', how='outer')

master = master.sort_values('year').reset_index(drop=True)
print('\nOuter-merged master:', master.shape)
print('Year range:', master['year'].min(), 'to', master['year'].max())

M1 mobility            : (16, 3)
M2 cpi                 : (66, 3)
M2 income              : (26, 3)
M2 gini                : (26, 2)
M2 gdp gap             : (66, 4)
M3 hdb price index     : (37, 2)
M3 hdb transactions    : (10, 3)
M4 WHR Singapore       : (16, 6)
M5 principal causes    : (17, 6)

Outer-merged master: (67, 24)
Year range: 1960 to 2026


In [16]:
# Trim to the agreed analysis window 2010-2025 (when most series start)
WINDOW_START = 2010
WINDOW_END   = 2025

master = master[(master['year'] >= WINDOW_START) & (master['year'] <= WINDOW_END)].reset_index(drop=True)
print(f'After windowing ({WINDOW_START}-{WINDOW_END}): {master.shape}')

master.to_csv(PROCESSED / 'master_yearly.csv', index=False)
print(f'Saved: master_yearly.csv ({(PROCESSED / "master_yearly.csv").stat().st_size / 1024:.1f} KB)')
master.head()

After windowing (2010-2025): (16, 24)
Saved: master_yearly.csv (3.3 KB)


,year,m1_mean_weekly_hours,m1_pct_40plus_hours,m2_cpi_all_items,m2_cpi_food,m2_median_household_income_sgd,m2_real_income_change_pct,m2_gini_coefficient,m2_gdp_per_capita_sgd,m2_income_per_person_sgd,m2_gdp_vs_income_gap_x,m3_hdb_resale_price_index,m3_hdb_median_resale_price,m3_hdb_resale_transactions,m4_life_ladder,m4_freedom_score,m4_social_support,m4_positive_affect,m4_negative_affect,m5_pct_deaths_cancer,m5_pct_deaths_cerebrovascular_disease_including_stroke,m5_pct_deaths_external_causes_of_morbidity_and_mortality,m5_pct_deaths_ischaemic_heart_disease,m5_pct_deaths_pneumonia
0,2010,46.6,88.3,75.099333,70.172083,6350.0,2.8,0.4485,64408.0,2041.800643,2.628725,118.600,NaN,NaN,6.531,0.846,0.864,0.527,0.131,28.5,8.4,NaN,18.7,15.7
1,2011,45.9,87.3,79.042917,72.314417,7040.0,5.5,0.4480,67783.0,2263.665595,2.495326,132.475,NaN,NaN,6.561,0.822,0.904,0.404,0.144,30.0,9.0,NaN,16.4,16.0
2,2012,45.6,87.4,82.663750,73.991250,7570.0,2.7,0.4550,69417.0,2434.083601,2.376562,142.150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.1,NaN,5.6,NaN,16.8
3,2013,45.3,87.1,84.618250,75.540583,7882.0,1.7,0.4360,71283.0,2534.405145,2.343844,147.975,NaN,NaN,6.533,0.827,0.808,0.663,0.148,30.5,NaN,4.9,NaN,18.5
4,2014,44.3,85.6,85.492500,77.726917,8297.0,4.0,0.4375,72938.0,2667.845659,2.278305,140.275,NaN,NaN,7.062,0.835,0.822,0.774,0.180,29.4,NaN,4.7,NaN,19.0


In [17]:
# Coverage check: how many years of data per source?
coverage = (
    master.set_index('year').notna().T
    .groupby(lambda c: c.split('_')[0] if '_' in c else c)
    .any().T.sum().sort_values(ascending=False)
)
print(f'Years of data per source (out of {len(master)} window-years):')
print(coverage.to_string())

Years of data per source (out of 16 window-years):
m1    16
m2    16
m3    16
m5    13
m4    12


## Section 7: Data Cleaning

### Per-dataset checklist

**M1 Work-Life**
- Check `m1_*` columns are numeric and not negative
- Mean weekly hours should sit in a plausible range (35 to 50)
- Percentage 40+ hours should sit in a plausible range (50 to 100)

**M2 Cost of Living**
- CPI values are positive and trending up
- Gini coefficient stays in the 0.3 to 0.55 band
- Median household income is in SGD per month (four-figure values), not yearly
- GDP per capita / income per person gap column already exists from the merge step

**M3 Housing**
- HDB Resale Price Index is positive
- HDB median resale price is in SGD (six figures)
- Note: M3 transactions only cover 2017 onwards (acceptable, flagged when charts use it)

**M4 World Happiness**
- Life Ladder stays in 0 to 10 band
- Note: 2012, 2020, 2024, 2025 are NaN because WHR did not survey Singapore those years (acceptable gap, flagged when charts use it)

**M5 Healthcare**
- Top-cause death percentages sum to a sensible value (well under 100)
- Cerebrovascular and ischaemic heart disease columns drop off after 2011 because MOH split/merged categories. Acceptable gap, flagged when used.

### Derived columns to add

- `m4_life_ladder_yoy_change` year-on-year change in Singapore's happiness score
- `m3_hdb_price_5y_change_pct` 5-year percent change in HDB Resale Price Index (affordability acceleration signal)
- `m2_income_growth_pct` year-on-year growth in median household income (compare against CPI growth)

These derived columns make cross-member charts easier later.

In [18]:
# 1. Check missing values in the windowed master
missing = master.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(missing.to_string() if len(missing) else '(none)')

Columns with missing values:
m5_pct_deaths_ischaemic_heart_disease                       14
m5_pct_deaths_cerebrovascular_disease_including_stroke      14
m3_hdb_median_resale_price                                   7
m3_hdb_resale_transactions                                   7
m5_pct_deaths_external_causes_of_morbidity_and_mortality     5
m4_positive_affect                                           4
m4_freedom_score                                             4
m4_life_ladder                                               4
m4_social_support                                            4
m4_negative_affect                                           4
m5_pct_deaths_cancer                                         3
m5_pct_deaths_pneumonia                                      3
m1_pct_40plus_hours                                          1


In [19]:
# 2. Check for duplicate years
duplicates = master['year'].duplicated().sum()
print(f'Duplicate years: {duplicates}')
if duplicates:
    master = master.drop_duplicates(subset='year').reset_index(drop=True)
    print('Duplicates removed.')

Duplicate years: 0


In [20]:
# 3. Make sure every column except `year` is numeric
for col in master.columns:
    if col != 'year':
        master[col] = pd.to_numeric(master[col], errors='coerce')

print('dtypes after coercion:')
print(master.dtypes.value_counts().to_string())

dtypes after coercion:
float64    23
Int64       1


In [21]:
# 4. Sanity check: negative values where they should not exist
should_be_positive = [
    'm1_mean_weekly_hours', 'm1_pct_40plus_hours',
    'm2_cpi_all_items', 'm2_cpi_food',
    'm2_median_household_income_sgd', 'm2_gini_coefficient',
    'm2_gdp_per_capita_sgd', 'm2_income_per_person_sgd',
    'm3_hdb_resale_price_index', 'm3_hdb_median_resale_price',
    'm4_life_ladder',
]
print('Negative-value check (these should ALL be 0):')
for c in should_be_positive:
    if c in master.columns:
        n = (master[c] < 0).sum()
        print(f'  {c:<45} {n}')

Negative-value check (these should ALL be 0):
  m1_mean_weekly_hours                          0
  m1_pct_40plus_hours                           0
  m2_cpi_all_items                              0
  m2_cpi_food                                   0
  m2_median_household_income_sgd                0
  m2_gini_coefficient                           0
  m2_gdp_per_capita_sgd                         0
  m2_income_per_person_sgd                      0
  m3_hdb_resale_price_index                     0
  m3_hdb_median_resale_price                    0
  m4_life_ladder                                0


In [22]:
# 5. Small gap-fill for short isolated NaNs (max 1 year forward, 1 year back)
# We do NOT fill long gaps because doing that would invent data that does not exist.
# Long gaps stay as NaN and the chart code disclose them with a caption.

fillable_cols = [
    'm2_cpi_all_items', 'm2_cpi_food',
    'm2_median_household_income_sgd', 'm2_gini_coefficient',
    'm2_gdp_per_capita_sgd', 'm2_income_per_person_sgd',
    'm3_hdb_resale_price_index',
    'm1_mean_weekly_hours', 'm1_pct_40plus_hours',
]
for col in fillable_cols:
    if col in master.columns:
        master[col] = master[col].ffill(limit=1).bfill(limit=1)
print('Small gaps filled in', len(fillable_cols), 'columns (max 1-year fill).')

Small gaps filled in 9 columns (max 1-year fill).


In [23]:
# 6. Create useful derived columns for cross-member charts

# m4 year-on-year happiness change (negative means happiness fell)
if 'm4_life_ladder' in master.columns:
    master['m4_life_ladder_yoy_change'] = master['m4_life_ladder'].diff()

# m3 5-year price acceleration (how fast resale prices are climbing)
if 'm3_hdb_resale_price_index' in master.columns:
    master['m3_hdb_price_5y_change_pct'] = master['m3_hdb_resale_price_index'].pct_change(5) * 100

# m2 income growth and CPI growth side by side
if 'm2_median_household_income_sgd' in master.columns:
    master['m2_income_growth_pct'] = master['m2_median_household_income_sgd'].pct_change() * 100
if 'm2_cpi_all_items' in master.columns:
    master['m2_cpi_growth_pct']    = master['m2_cpi_all_items'].pct_change() * 100

# Headline reference for cross-member charts: youth suicide share 2024 (single number)
# This is computed from a side-table earlier in the notebook. We add it as a constant
# column so M5 can plot it easily next to anyone else's year-by-year series.
youth_2024 = 25.3  # already computed in Section 5, kept here for reference
master['m5_youth_suicide_share_2024_ref'] = youth_2024

print('Derived columns added:')
for c in ['m4_life_ladder_yoy_change', 'm3_hdb_price_5y_change_pct',
          'm2_income_growth_pct', 'm2_cpi_growth_pct',
          'm5_youth_suicide_share_2024_ref']:
    if c in master.columns:
        nn = master[c].notna().sum()
        print(f'  {c:<35} {nn}/{len(master)} non-null')

Derived columns added:
  m4_life_ladder_yoy_change           9/16 non-null
  m3_hdb_price_5y_change_pct          11/16 non-null
  m2_income_growth_pct                15/16 non-null
  m2_cpi_growth_pct                   15/16 non-null
  m5_youth_suicide_share_2024_ref     16/16 non-null


In [24]:
# 7. Final check after cleaning
print('Final master shape:', master.shape)
print('Year range:', int(master['year'].min()), 'to', int(master['year'].max()))

remaining = master.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print('\nRemaining missing values:')
print(remaining.to_string() if len(remaining) else '(none)')

Final master shape: (16, 29)
Year range: 2010 to 2025

Remaining missing values:
m5_pct_deaths_ischaemic_heart_disease                       14
m5_pct_deaths_cerebrovascular_disease_including_stroke      14
m3_hdb_median_resale_price                                   7
m3_hdb_resale_transactions                                   7
m4_life_ladder_yoy_change                                    7
m5_pct_deaths_external_causes_of_morbidity_and_mortality     5
m3_hdb_price_5y_change_pct                                   5
m4_life_ladder                                               4
m4_freedom_score                                             4
m4_negative_affect                                           4
m4_positive_affect                                           4
m4_social_support                                            4
m5_pct_deaths_cancer                                         3
m5_pct_deaths_pneumonia                                      3
m2_income_growth_pct                 

In [25]:
# 8. Save the cleaned master
master_clean = master.copy()
out = PROCESSED / 'master_clean.csv'
master_clean.to_csv(out, index=False)
print(f'Saved cleaned dataset as {out.name} ({out.stat().st_size/1024:.1f} KB)')
print(f'Cleaned shape: {master_clean.shape}')

# Reminder: from Section 9 onwards, every member should load master_clean.csv (NOT master_yearly.csv)
print('\nFrom Section 9 onwards, load master_clean.csv (not master_yearly.csv).')

Saved cleaned dataset as master_clean.csv (4.4 KB)
Cleaned shape: (16, 29)

From Section 9 onwards, load master_clean.csv (not master_yearly.csv).


## Section 8: Final inventory of processed files

In [26]:
print('Processed files saved to data/processed/:')
for f in sorted(PROCESSED.glob('*.csv')):
    print(f'  {f.name:<48} {f.stat().st_size/1024:>10,.1f} KB')

Processed files saved to data/processed/:
  cause_of_death_2024_long.csv                          120.4 KB
  cpi_monthly.csv                                       257.3 KB
  hdb_price_index_quarterly.csv                           2.8 KB
  hdb_resale_monthly.csv                             23,340.8 KB
  master_clean.csv                                        4.4 KB
  master_yearly.csv                                       3.3 KB
  mccy_youth_happiness.csv                                0.7 KB
  sleep_health_global.csv                                22.8 KB
  suicide_share_by_age_2024.csv                           0.6 KB
  whr_2024_ranking.csv                                   12.8 KB
  whr_panel_full.csv                                    157.5 KB


## Section 9: What each member uses

| Member | Primary file | Side-table |
|---|---|---|
| M1 Work-Life | `master_clean.csv` (cols `m1_*`) | — |
| M2 Cost of Living | `master_clean.csv` (cols `m2_*`) | `cpi_monthly.csv` |
| M3 Housing | `master_clean.csv` (cols `m3_*`) | `hdb_price_index_quarterly.csv`, `hdb_resale_monthly.csv` |
| M4 Sleep / Happiness | `master_clean.csv` (cols `m4_*`) | `whr_panel_full.csv`, `whr_2024_ranking.csv`, `sleep_health_global.csv` |
| M5 Youth/Healthcare | `master_clean.csv` (cols `m5_*`) | `suicide_share_by_age_2024.csv`, `cause_of_death_2024_long.csv`, `mccy_youth_happiness.csv` |

**Important:** From this section onwards, always load `master_clean.csv` (not `master_yearly.csv`). The clean version has the derived columns and any small gaps filled.